In [1]:

# LAB 10 - Stacking Classifier, Feature Correlation Heatmap, PCA (99% variance)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, RandomizedSearchCV,
                                     cross_val_score, StratifiedKFold)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              StackingClassifier, GradientBoostingClassifier)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.exceptions import ConvergenceWarning

# --- silence benign warnings -------------------------------------------------
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", message=".*IProgress not found.*")
warnings.filterwarnings("ignore", message=".*select_dtypes.*")
warnings.filterwarnings("ignore", message=".*probability.*deprecated.*")

# optional boosters
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except Exception:
    HAS_CAT = False


# =============================================================================
# 1. DATA LOADING & CLEANING  (identical to Lab 09)
# =============================================================================
def load_data(path, sheet_name="marketing_campaign"):
    """Load and clean the marketing campaign sheet."""
    df = pd.read_excel(path, sheet_name=sheet_name)
    df.columns = [c.strip() for c in df.columns]

    # Fix mixed datetime/str in Dt_Customer
    if "Dt_Customer" in df.columns:
        dt = pd.to_datetime(df["Dt_Customer"], errors="coerce", dayfirst=True)
        df["Cust_Year"]  = dt.dt.year
        df["Cust_Month"] = dt.dt.month
        df["Cust_Day"]   = dt.dt.day
        df["Cust_Date_Missing"] = dt.isna().astype(int)
        df = df.drop(columns=["Dt_Customer"])

    drop_cols = [c for c in ["ID", "Z_CostContact", "Z_Revenue"] if c in df.columns]
    df = df.drop(columns=drop_cols)

    df["Income"]   = pd.to_numeric(df["Income"], errors="coerce")
    df["Response"] = df["Response"].astype(int)

    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype(str)

    return df


def split_features_target(df, target="Response"):
    """Separate X and y."""
    return df.drop(columns=[target]), df[target]


def build_preprocessor(X):
    """ColumnTransformer for numeric + categorical features."""
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot",  OneHotEncoder(handle_unknown="ignore")),
    ])
    pre = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])
    return pre, num_cols, cat_cols


def evaluate_model(model, X_test, y_test):
    """Standard classification metrics."""
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)
    return {
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall":    recall_score(y_test, y_pred, zero_division=0),
        "F1":        f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC":   roc_auc_score(y_test, y_score),
        "Confusion": confusion_matrix(y_test, y_pred),
    }


# =============================================================================
# 2. A1 - STACKING CLASSIFIER
# =============================================================================
def build_base_estimators(seed=42):
    """
    Base learners re-used from Lab 09.
    Each returns a bare classifier (no preprocessing — the outer
    StackingClassifier pipeline handles that via a shared preprocessor).
    """
    base = {
        "Perceptron":   Perceptron(max_iter=1000, random_state=seed),
        "MLP":          MLPClassifier(hidden_layer_sizes=(64, 32),
                                      max_iter=500, random_state=seed),
        "SVM":          CalibratedClassifierCV(SVC(random_state=seed),
                                               ensemble=False, cv=3),
        "DecisionTree": DecisionTreeClassifier(random_state=seed),
        "RandomForest": RandomForestClassifier(n_estimators=200,
                                               random_state=seed, n_jobs=-1),
        "NaiveBayes":   GaussianNB(),
    }
    if HAS_XGB:
        base["XGBoost"] = XGBClassifier(eval_metric="logloss",
                                        random_state=seed, n_jobs=-1)
    if HAS_CAT:
        base["CatBoost"] = CatBoostClassifier(verbose=0, random_state=seed)
    return base


def build_stacking_pipeline(preprocessor, base_estimators,
                            final_estimator, cv=5, passthrough=False):
    """
    Compose preprocessing + stacking (base learners -> meta-model).
    Wrapped in a single Pipeline so fit/predict behave like any classifier.
    """
    stack = StackingClassifier(
        estimators=list(base_estimators.items()),
        final_estimator=final_estimator,
        cv=cv,
        passthrough=passthrough,   # True = give meta-model raw features too
        n_jobs=-1,
    )
    return Pipeline([("preprocessing", preprocessor), ("stacking", stack)])


def experiment_stacking(preprocessor, base_estimators,
                        X_train, y_train, X_test, y_test):
    """
    Try several meta-models and return a comparison DataFrame.
    Meta-models chosen to cover linear, regularised, probabilistic,
    and non-linear behaviour.
    """
    meta_models = {
        "LogReg":            LogisticRegression(max_iter=1000, random_state=42),
        "LogReg_L2_C0.1":    LogisticRegression(C=0.1, max_iter=1000,
                                                random_state=42),
        "MLP_meta":          MLPClassifier(hidden_layer_sizes=(32,),
                                           max_iter=500, random_state=42),
        "RandomForest_meta": RandomForestClassifier(n_estimators=100,
                                                    random_state=42, n_jobs=-1),
        "GNB_meta":          GaussianNB(),
        "KNN_meta":          KNeighborsClassifier(n_neighbors=5),
    }
    rows = []
    fitted = {}
    for meta_name, meta in meta_models.items():
        pipe = build_stacking_pipeline(preprocessor, base_estimators,
                                       final_estimator=meta, cv=5,
                                       passthrough=False)
        pipe.fit(X_train, y_train)
        m = evaluate_model(pipe, X_test, y_test)
        rows.append({
            "MetaModel": meta_name,
            "Accuracy":  m["Accuracy"],
            "Precision": m["Precision"],
            "Recall":    m["Recall"],
            "F1":        m["F1"],
            "ROC_AUC":   m["ROC_AUC"],
        })
        fitted[meta_name] = pipe
    return (pd.DataFrame(rows).sort_values("F1", ascending=False)
              .reset_index(drop=True), fitted)


def experiment_stacking_passthrough(preprocessor, base_estimators,
                                    X_train, y_train, X_test, y_test,
                                    meta=None):
    """Variant: give the meta-model the raw features too (passthrough=True)."""
    if meta is None:
        meta = LogisticRegression(max_iter=1000, random_state=42)
    pipe = build_stacking_pipeline(preprocessor, base_estimators,
                                   final_estimator=meta, cv=5,
                                   passthrough=True)
    pipe.fit(X_train, y_train)
    m = evaluate_model(pipe, X_test, y_test)
    return pipe, m


# =============================================================================
# 3. A2 - FEATURE CORRELATION ANALYSIS
# =============================================================================
def build_correlation_matrix(df, target="Response"):
    """
    Compute Pearson correlation of numeric columns + target.
    Non-numeric columns are dropped (they'll be one-hot encoded downstream).
    """
    numeric_df = df.select_dtypes(include=[np.number]).copy()
    # Median-impute Income purely for the correlation study; the actual
    # modelling pipeline still uses the proper imputer.
    numeric_df = numeric_df.fillna(numeric_df.median())
    corr = numeric_df.corr(method="pearson")
    return corr, numeric_df


def plot_correlation_heatmap(corr, out_path="correlation_heatmap.png",
                             figsize=(16, 12)):
    """Draw the full correlation heatmap and save it."""
    plt.figure(figsize=figsize)
    sns.heatmap(corr, cmap="coolwarm", center=0, annot=False,
                square=True, linewidths=0.4)
    plt.title("Pearson correlation matrix - marketing_campaign numeric features")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()


def top_correlated_pairs(corr, target="Response", k=10):
    """
    Return the top-k absolute (feature, target) correlations
    and the top-k absolute feature-feature pairs (off-diagonal).
    """
    # ---- Target correlations
    tgt = (corr[target].drop(target)
                       .abs()
                       .sort_values(ascending=False)
                       .head(k)
                       .reset_index())
    tgt.columns = ["Feature", "Abs_Corr_with_Target"]

    # ---- Feature-feature pairs
    cc = corr.drop(index=target, columns=target)
    pairs = []
    cols = cc.columns.tolist()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            pairs.append((cols[i], cols[j], cc.iloc[i, j]))
    pairs_df = (pd.DataFrame(pairs, columns=["Feature_1", "Feature_2", "Corr"])
                  .assign(Abs_Corr=lambda d: d["Corr"].abs())
                  .sort_values("Abs_Corr", ascending=False)
                  .head(k)
                  .reset_index(drop=True))
    return tgt, pairs_df


# =============================================================================
# 4. A3 - PCA with 99% CUSUM-explained variance
# =============================================================================
def pca_cusum_analysis(preprocessor, X_train):
    """
    Fit preprocessor, then PCA on the training set and return:
      * full PCA object (with all components)
      * number of components needed for 99% variance
      * cumulative explained-variance array
    'Cusum variance' = cumulative-sum of explained_variance_ratio_.
    """
    X_t = preprocessor.fit_transform(X_train)
    if hasattr(X_t, "toarray"):
        X_t = X_t.toarray()

    pca_full = PCA(n_components=None, random_state=42).fit(X_t)
    cusum    = np.cumsum(pca_full.explained_variance_ratio_)
    n_99     = int(np.searchsorted(cusum, 0.99) + 1)   # first index where >= 0.99
    return pca_full, cusum, n_99, X_t


def plot_pca_cusum(cusum, n_99, out_path="pca_cusum.png"):
    """Save the CUSUM (cumulative explained variance) curve."""
    plt.figure(figsize=(9, 5))
    plt.plot(np.arange(1, len(cusum) + 1), cusum, marker=".", markersize=4)
    plt.axhline(0.99, color="red", linestyle="--", label="99% threshold")
    plt.axvline(n_99, color="green", linestyle="--",
                label=f"{n_99} components")
    plt.xlabel("Number of components")
    plt.ylabel("Cumulative explained variance (CUSUM)")
    plt.title("PCA cumulative explained variance")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()


def build_pca_pipeline(preprocessor, n_components, classifier):
    """
    Pipeline: preprocessing -> PCA(n_99) -> classifier.
    Uses whiten=True so downstream models (SVM / MLP / Perceptron) get
    unit-variance principal components — usually better numerically.
    """
    return Pipeline([
        ("preprocessing", preprocessor),
        ("pca",           PCA(n_components=n_components, whiten=True,
                              random_state=42)),
        ("classifier",    classifier),
    ])


def run_all_classifiers_on_pca(preprocessor, n_components,
                               base_estimators, X_train, y_train,
                               X_test, y_test):
    """Fit each base classifier on PCA-transformed features and return metrics."""
    rows, fitted = [], {}
    for name, clf in base_estimators.items():
        pipe = build_pca_pipeline(preprocessor, n_components, clf)
        pipe.fit(X_train, y_train)
        m = evaluate_model(pipe, X_test, y_test)
        rows.append({
            "Model":     name,
            "Accuracy":  m["Accuracy"],
            "Precision": m["Precision"],
            "Recall":    m["Recall"],
            "F1":        m["F1"],
            "ROC_AUC":   m["ROC_AUC"],
        })
        fitted[name] = pipe
    return (pd.DataFrame(rows).sort_values("F1", ascending=False)
              .reset_index(drop=True), fitted)


def run_all_classifiers_no_pca(preprocessor, base_estimators,
                               X_train, y_train, X_test, y_test):
    """Same classifiers but WITHOUT PCA — for the comparison table."""
    rows, fitted = [], {}
    for name, clf in base_estimators.items():
        pipe = Pipeline([("preprocessing", preprocessor),
                         ("classifier",    clf)])
        pipe.fit(X_train, y_train)
        m = evaluate_model(pipe, X_test, y_test)
        rows.append({
            "Model":     name,
            "Accuracy":  m["Accuracy"],
            "Precision": m["Precision"],
            "Recall":    m["Recall"],
            "F1":        m["F1"],
            "ROC_AUC":   m["ROC_AUC"],
        })
        fitted[name] = pipe
    return (pd.DataFrame(rows).sort_values("F1", ascending=False)
              .reset_index(drop=True), fitted)


def compare_pca_vs_no_pca(no_pca_df, pca_df):
    """Side-by-side comparison for the report."""
    left  = no_pca_df.set_index("Model")[["F1", "ROC_AUC"]]
    right = pca_df.set_index("Model")[["F1", "ROC_AUC"]]
    merged = left.join(right, lsuffix="_noPCA", rsuffix="_PCA")
    merged["ΔF1"]     = merged["F1_PCA"]      - merged["F1_noPCA"]
    merged["ΔROC_AUC"] = merged["ROC_AUC_PCA"] - merged["ROC_AUC_noPCA"]
    return merged.reset_index()


# =============================================================================
# MAIN PROGRAM
# =============================================================================
if __name__ == "__main__":
    DATA_PATH = r"D:\C drive files\Downloads\Lab Session Data (1) (1).xlsx"
    SHEET     = "marketing_campaign"

    # ---- Load / split / preprocessor
    df = load_data(DATA_PATH, SHEET)
    X, y = split_features_target(df, target="Response")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    preprocessor, num_cols, cat_cols = build_preprocessor(X_train)

    print("=== Data Overview ===")
    print(f"Rows: {len(df)}, Features: {X.shape[1]}, Positives: {int(y.sum())}")
    print(f"Numeric cols: {len(num_cols)}, Categorical cols: {len(cat_cols)}\n")

    # =========================================================================
    # A2 - Correlation analysis (do this first — cheap, and it is discussed in
    # the report before the stacking / PCA results)
    # =========================================================================
    print("=== A2: Feature Correlation Analysis ===")
    corr, numeric_df = build_correlation_matrix(df, target="Response")
    plot_correlation_heatmap(corr, out_path="correlation_heatmap.png")
    print("Correlation heatmap saved: 'correlation_heatmap.png'")

    tgt_corr, pair_corr = top_correlated_pairs(corr, target="Response", k=10)
    print("\nTop-10 absolute |corr(feature, Response)|:")
    print(tgt_corr.to_string(index=False))
    print("\nTop-10 absolute |corr(feature_i, feature_j)| (highly collinear pairs):")
    print(pair_corr[["Feature_1", "Feature_2", "Corr"]].to_string(index=False))
    print()

    # =========================================================================
    # A1 - Stacking classifier with different meta-models
    # =========================================================================
    print("=== A1: Stacking Classifier (multiple meta-models) ===")
    base_estimators = build_base_estimators()
    print(f"Base learners: {list(base_estimators.keys())}")
    stacking_df, stacking_fitted = experiment_stacking(
        preprocessor, base_estimators,
        X_train, y_train, X_test, y_test,
    )
    print("\nMeta-model comparison (stacking, cv=5, passthrough=False):")
    print(stacking_df.to_string(index=False))

    # Baseline: best base learner alone (for context)
    print("\n--- Best single base learner (for comparison) ---")
    single_rows = []
    for name, clf in base_estimators.items():
        pipe = Pipeline([("preprocessing", preprocessor), ("classifier", clf)])
        pipe.fit(X_train, y_train)
        m = evaluate_model(pipe, X_test, y_test)
        single_rows.append({"Model": name, "F1": m["F1"], "ROC_AUC": m["ROC_AUC"]})
    single_df = pd.DataFrame(single_rows).sort_values("F1", ascending=False).reset_index(drop=True)
    print(single_df.to_string(index=False))

    # Passthrough variant (meta-model also sees raw features)
    pipe_pt, m_pt = experiment_stacking_passthrough(
        preprocessor, base_estimators,
        X_train, y_train, X_test, y_test,
    )
    print("\nStacking with passthrough=True (meta=LogReg): "
          f"F1={m_pt['F1']:.4f}, ROC-AUC={m_pt['ROC_AUC']:.4f}")
    print("Confusion matrix:\n", m_pt["Confusion"], "\n")

    # =========================================================================
    # A3 - PCA (retain 99% CUSUM variance), then classifiers
    # =========================================================================
    print("=== A3: PCA (99% CUSUM explained variance) ===")
    pca_full, cusum, n_99, X_train_t = pca_cusum_analysis(preprocessor, X_train)
    plot_pca_cusum(cusum, n_99, out_path="pca_cusum.png")
    print(f"Original (post-preprocessing) feature count: {X_train_t.shape[1]}")
    print(f"Components required for 99% CUSUM variance:  {n_99}")
    print(f"Total explained variance with {n_99} components: "
          f"{cusum[n_99-1]:.4f}")
    print("PCA CUSUM plot saved: 'pca_cusum.png'\n")

    # Classifiers WITHOUT PCA (baseline, same split)
    print("--- Classifiers WITHOUT PCA ---")
    no_pca_df, _ = run_all_classifiers_no_pca(
        preprocessor, build_base_estimators(),
        X_train, y_train, X_test, y_test,
    )
    print(no_pca_df.to_string(index=False))

    # Classifiers WITH PCA
    print("\n--- Classifiers WITH PCA (n_99 components) ---")
    pca_df, _ = run_all_classifiers_on_pca(
        preprocessor, n_99, build_base_estimators(),
        X_train, y_train, X_test, y_test,
    )
    print(pca_df.to_string(index=False))

    # Comparison
    print("\n--- PCA vs No-PCA (Δ = PCA − NoPCA) ---")
    comparison = compare_pca_vs_no_pca(no_pca_df, pca_df)
    print(comparison.to_string(index=False))

    # Stacking on PCA (final combined result)
    print("\n--- Stacking Classifier on PCA features (meta=LogReg) ---")
    pipe_pca_stack = Pipeline([
        ("preprocessing", preprocessor),
        ("pca",           PCA(n_components=n_99, whiten=True, random_state=42)),
        ("stacking",      StackingClassifier(
                              estimators=list(base_estimators.items()),
                              final_estimator=LogisticRegression(max_iter=1000,
                                                                 random_state=42),
                              cv=5, n_jobs=-1)),
    ])
    pipe_pca_stack.fit(X_train, y_train)
    m = evaluate_model(pipe_pca_stack, X_test, y_test)
    print(f"Stacking+PCA: Acc={m['Accuracy']:.4f}, F1={m['F1']:.4f}, "
          f"ROC-AUC={m['ROC_AUC']:.4f}")
    print("Confusion matrix:\n", m["Confusion"])

=== Data Overview ===
Rows: 2240, Features: 28, Positives: 334
Numeric cols: 26, Categorical cols: 2

=== A2: Feature Correlation Analysis ===
Correlation heatmap saved: 'correlation_heatmap.png'

Top-10 absolute |corr(feature, Response)|:
            Feature  Abs_Corr_with_Target
       AcceptedCmp5              0.326634
       AcceptedCmp1              0.293982
       AcceptedCmp3              0.254258
           MntWines              0.247254
    MntMeatProducts              0.236335
NumCatalogPurchases              0.220810
            Recency              0.198437
       AcceptedCmp4              0.177019
          Cust_Year              0.171030
       AcceptedCmp2              0.169293

Top-10 absolute |corr(feature_i, feature_j)| (highly collinear pairs):
      Feature_1           Feature_2     Corr
MntMeatProducts NumCatalogPurchases 0.723827
       MntWines   NumStorePurchases 0.642100
       MntWines NumCatalogPurchases 0.635226
      MntFruits     MntFishProducts 0.594804
 